### 1. Scale Estimator Speed Benchmark (AbsMean vs AbsMean-NoZero vs RMS)

In [1]:
import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)
device = torch.device('cuda')

# -----------------------------------------------------------------------
# Match real model dimensions -- largest ternary matrix is MLP fc = [3072, 768].
# GROUP_SIZE=128 matches the submission config (BITNET_GROUP_SIZE=128).
# -----------------------------------------------------------------------

W          = torch.randn(3072, 768, device=device, dtype=torch.bfloat16)
GROUP_SIZE = 128
N_WARMUP   = 200
N_RUNS     = 2000


def ste_absmean(w):
    """Current scheme: absmean over all weights including zeros."""
    g     = GROUP_SIZE
    w_g   = w.reshape(-1, g)
    scale = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    q     = (w_g / scale).round().clamp_(-1, 1)
    return w + ((q * scale).view_as(w) - w).detach()


def ste_absmean_nozero(w):
    """Alternative 1: absmean over non-zero weights only.
    Eliminates shrinkage correction at load time entirely since
    q_absmean of non-zero {-1,+1} subset is always 1.0 by construction."""
    g        = GROUP_SIZE
    w_g      = w.reshape(-1, g)
    scale    = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    q        = (w_g / scale).round().clamp_(-1, 1)
    abs_w    = w_g.abs()
    nz_sum   = abs_w.sum(-1, keepdim=True)
    nz_count = (abs_w > 0).float().sum(-1, keepdim=True).clamp(min=1)
    scale_nz = (nz_sum / nz_count).clamp(min=1e-8)
    return w + ((q * scale_nz).view_as(w) - w).detach()


def ste_rms(w):
    """Alternative 2: RMS scale. Theoretically optimal under squared error.
    Correction factor at load is scale/q_rms = 1/sqrt(1-zero_frac),
    grows much slower than absmean's 1/(1-zero_frac) at high sparsity."""
    g     = GROUP_SIZE
    w_g   = w.reshape(-1, g)
    scale = w_g.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-8)
    q     = (w_g / scale).round().clamp_(-1, 1)
    return w + ((q * scale).view_as(w) - w).detach()


for _ in range(N_WARMUP):
    _ = ste_absmean(W)
    _ = ste_absmean_nozero(W)
    _ = ste_rms(W)
torch.cuda.synchronize()

timings = {}
for name, fn in [
    ('AbsMean (current)',  ste_absmean),
    ('AbsMean-NoZero',     ste_absmean_nozero),
    ('RMS',                ste_rms),
]:
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(W)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    timings[name] = ms
    print(f'{name:<25} {ms:.4f} ms/call')

# 10L model: c_qkv(1) + attn_proj(1) + mlp_fc(1) + mlp_proj(1) = 4 STE calls per block.
# 10 blocks = 40 STE calls per step.
print()
for name, ms in timings.items():
    print(f'{name:<25} 40 STE calls/step (10L): {40 * ms:.3f} ms overhead')

AbsMean (current)         0.0592 ms/call
AbsMean-NoZero            0.1138 ms/call
RMS                       0.0636 ms/call

AbsMean (current)         40 STE calls/step (10L): 2.368 ms overhead
AbsMean-NoZero            40 STE calls/step (10L): 4.552 ms overhead
RMS                       40 STE calls/step (10L): 2.544 ms overhead


### 2. Train a Small Ternary Model and Collect Checkpoints

Zero_frac emerges naturally from training dynamics -- same mechanism as the real model.
Architecture mirrors the submission: ternary linears, relu2 MLP, group_size=128.
Checkpoints collected at steps that bracket the real training milestones.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device     = torch.device('cuda')
GROUP_SIZE = 128

# -----------------------------------------------------------------------
# Small ternary MLP matching submission architecture proportions:
# dim=256, mlp_mult=4, group_size=128.
# Two blocks: each has an attention-proxy linear pair + relu2 MLP.
# Total ~1.3M ternary params -- fast to train, realistic distributions.
# -----------------------------------------------------------------------

DIM    = 256
HIDDEN = DIM * 4   # MLP_MULT=4


class TernaryLinear(nn.Linear):
    """BitNet-style ternary linear with absmean STE. Matches TernaryLinear
    in the submission -- same group_size, same quantization formula."""
    def __init__(self, in_f, out_f, group_size=128):
        super().__init__(in_f, out_f, bias=False)
        self.group_size = group_size

    def forward(self, x):
        w   = self.weight.bfloat16()
        g   = self.group_size
        w_g = w.reshape(-1, g)
        s   = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
        q   = (w_g / s).round().clamp(-1, 1)
        w_t = w + ((q * s).reshape(w.shape) - w).detach()
        return F.linear(x, w_t)


class TernaryBlock(nn.Module):
    """One transformer-like block: linear pair (proxy for attn) + relu2 MLP.
    Uses RMSNorm and residual connections matching the submission."""
    def __init__(self, dim, hidden, group_size):
        super().__init__()
        # Attention proxy: two linears (qkv collapsed to one, plus output proj)
        self.attn_in  = TernaryLinear(dim, dim,    group_size)
        self.attn_out = TernaryLinear(dim, dim,    group_size)
        # MLP: fc (gate+up fused) + proj, relu2 activation
        self.mlp_fc   = TernaryLinear(dim, hidden, group_size)
        self.mlp_proj = TernaryLinear(hidden, dim, group_size)

    def forward(self, x):
        # Attn-proxy sublayer
        h = F.rms_norm(x, (x.shape[-1],))
        x = x + self.attn_out(F.relu(self.attn_in(h)))
        # MLP sublayer with relu2 matching the submission
        h = F.rms_norm(x, (x.shape[-1],))
        x = x + self.mlp_proj(torch.relu(self.mlp_fc(h)).square())
        return x


class TernaryModel(nn.Module):
    def __init__(self, dim, hidden, n_blocks, group_size):
        super().__init__()
        self.blocks = nn.ModuleList([
            TernaryBlock(dim, hidden, group_size) for _ in range(n_blocks)
        ])
        self.head = nn.Linear(dim, dim, bias=False)  # fp output head

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return self.head(x)


model = TernaryModel(DIM, HIDDEN, n_blocks=2, group_size=GROUP_SIZE).to(device).bfloat16()
opt   = torch.optim.Adam(model.parameters(), lr=3e-4)

n_params = sum(p.numel() for n, p in model.named_parameters()
               if 'block' in n and p.ndim == 2)
print(f'Ternary params: {n_params:,}  (dim={DIM}, hidden={HIDDEN}, 2 blocks)')

# Fixed random regression dataset -- repeating batches, same as real training
torch.manual_seed(0)
X_data = torch.randn(512, DIM, device=device, dtype=torch.bfloat16)
Y_data = torch.randn(512, DIM, device=device, dtype=torch.bfloat16)

# Checkpoints bracketing real training milestones:
# 100   = early warmup
# 1000  = early stable training
# 6500  = approx 10-min submission step count
# 25000 = mid long run
# 50000 = binary submission step count
CHECKPOINTS = [0, 100, 500, 1000, 2500, 5000, 10000, 25000, 50000]
MAX_STEPS   = max(CHECKPOINTS)
saved_weights = {}  # step -> {name: float32 cpu tensor}


def save_weights(model, step):
    saved_weights[step] = {
        name: p.detach().float().cpu().clone()
        for name, p in model.named_parameters()
        if p.ndim == 2 and p.numel() >= GROUP_SIZE * 4  # only ternary-sized matrices
        and 'head' not in name
    }


save_weights(model, 0)

for step in range(1, MAX_STEPS + 1):
    start = ((step - 1) * 64) % (X_data.shape[0] - 64)
    x     = X_data[start:start + 64]
    y     = Y_data[start:start + 64]

    opt.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(x), y)
    loss.backward()
    opt.step()

    if step in CHECKPOINTS:
        save_weights(model, step)
        # Quick zero_frac snapshot from first ternary matrix
        w0 = next(iter(saved_weights[step].values()))
        wg = w0.reshape(-1, GROUP_SIZE)
        s  = wg.abs().mean(-1, keepdim=True).clamp(min=1e-8)
        q  = (wg / s).round().clamp(-1, 1)
        zf = (q == 0).float().mean().item()
        print(f'step:{step:>6}  loss:{loss.item():.4f}  zero_frac:{zf:.3f}')

print(f'\nDone. {len(saved_weights)} checkpoints, {len(next(iter(saved_weights.values())))} matrices each.')

Ternary params: 1,310,720  (dim=256, hidden=1024, 2 blocks)
step:   100  loss:0.7852  zero_frac:0.252
step:   500  loss:0.1318  zero_frac:0.243
step:  1000  loss:0.0559  zero_frac:0.226
step:  2500  loss:0.0226  zero_frac:0.198
step:  5000  loss:0.0057  zero_frac:0.184
step: 10000  loss:0.0045  zero_frac:0.192
step: 25000  loss:0.0040  zero_frac:0.191
step: 50000  loss:0.0038  zero_frac:0.190

Done. 9 checkpoints, 8 matrices each.


### 3. Roundtrip Error vs Training Steps (All Schemes)

In [3]:
import torch
import numpy as np

# -----------------------------------------------------------------------
# For each checkpoint, quantize every ternary weight matrix with each
# (scale_fn, correction_fn) pair, reconstruct, measure mean absolute error.
# Zero_frac is measured from the standard absmean quantization since that
# is what the training-time STE uses regardless of scheme.
# -----------------------------------------------------------------------

GROUP_SIZE = 128


def scale_absmean(w_g):
    return w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)

def scale_absmean_nozero(w_g):
    abs_w    = w_g.abs()
    nz_sum   = abs_w.sum(-1, keepdim=True)
    nz_count = (abs_w > 0).float().sum(-1, keepdim=True).clamp(min=1)
    return (nz_sum / nz_count).clamp(min=1e-8)

def scale_rms(w_g):
    return w_g.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-8)


def correct_absmean_fp16(q, scale):
    """Original: FP16 stored scale + shrinkage correction."""
    scale_stored = scale.half().float()
    q_absmean    = q.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    return q * (scale_stored / q_absmean)

def correct_absmean_bf16(q, scale):
    """Current: BF16 stored scale + shrinkage correction.
    BF16 exponent range = FP32, so magnitude rounding is negligible."""
    scale_stored = scale.bfloat16().float()
    q_absmean    = q.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    return q * (scale_stored / q_absmean)

def correct_absmean_fp32(q, scale):
    """Ideal absmean: FP32 scale + shrinkage correction. Theoretical upper bound."""
    q_absmean = q.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    return q * (scale / q_absmean)

def correct_nozero(q, scale):
    """No correction: scale was computed from non-zero weights only,
    q_absmean for the {-1,+1} subset is always exactly 1.0."""
    return q * scale

def correct_rms_fp32(q, scale):
    """RMS: correction = 1/sqrt(1-zero_frac), grows much slower than
    absmean's 1/(1-zero_frac) as the model becomes sparser."""
    q_rms = q.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-8)
    return q * (scale / q_rms)


schemes = [
    ('AbsMean+FP16 (original)', scale_absmean,        correct_absmean_fp16),
    ('AbsMean+BF16 (current)',  scale_absmean,        correct_absmean_bf16),
    ('AbsMean+FP32 (ideal)',    scale_absmean,        correct_absmean_fp32),
    ('AbsMean-NoZero',          scale_absmean_nozero, correct_nozero),
    ('RMS+FP32',                scale_rms,            correct_rms_fp32),
]


def measure_checkpoint(weights_dict, scale_fn, correct_fn, group_size):
    """Quantize all matrices at this checkpoint and return
    (mean_zero_frac, mean_abs_error, max_abs_error) averaged across matrices."""
    all_zf, all_me, all_mx = [], [], []
    for w_flat in weights_dict.values():
        w_g      = w_flat.float().reshape(-1, group_size)
        # Quantization threshold always from absmean (matches training STE)
        scale_q  = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
        q        = (w_g / scale_q).round().clamp(-1, 1)
        zf       = (q == 0).float().mean().item()
        # Stored scale and correction vary by scheme
        scale    = scale_fn(w_g)
        recon    = correct_fn(q, scale)
        err      = (recon - w_g).abs()
        all_zf.append(zf)
        all_me.append(err.mean().item())
        all_mx.append(err.max().item())
    return np.mean(all_zf), np.mean(all_me), np.mean(all_mx)


all_results = {}  # step -> {scheme: (zf, mean_err, max_err)}

print(f'{"Steps":>7}  {"zero_frac":>9}', end='')
for name, _, _ in schemes:
    print(f'  {name:>25}', end='')
print()
print('-' * (20 + len(schemes) * 27))

for step in CHECKPOINTS:
    all_results[step] = {}
    zf_ref = None
    line   = f'{step:>7}'
    for i, (name, sfn, cfn) in enumerate(schemes):
        zf, me, mx = measure_checkpoint(saved_weights[step], sfn, cfn, GROUP_SIZE)
        all_results[step][name] = (zf, me, mx)
        if i == 0:
            zf_ref = zf
            line  += f'  {zf_ref:>9.4f}'
        line += f'  {me:.6f}'
    print(line)

  Steps  zero_frac    AbsMean+FP16 (original)     AbsMean+BF16 (current)       AbsMean+FP32 (ideal)             AbsMean-NoZero                   RMS+FP32
-----------------------------------------------------------------------------------------------------------------------------------------------------------
      0     0.2480  0.009452  0.009452  0.009452  0.010212  0.009455
    100     0.2543  0.010266  0.010266  0.010266  0.010873  0.010293
    500     0.2479  0.011648  0.011648  0.011648  0.012295  0.011694
   1000     0.2329  0.011774  0.011774  0.011774  0.012339  0.011820
   2500     0.2085  0.011937  0.011936  0.011937  0.012291  0.012002
   5000     0.1966  0.012161  0.012161  0.012161  0.012377  0.012227
  10000     0.2007  0.012502  0.012501  0.012502  0.012625  0.012512
  25000     0.2003  0.012653  0.012653  0.012653  0.012733  0.012646
  50000     0.1997  0.012708  0.012708  0.012708  0.012771  0.012696


### 4. Error Growth and Relative Performance Summary

In [4]:
# -----------------------------------------------------------------------
# Two views:
#   (a) Each scheme relative to FP16 baseline at the same step count
#   (b) Error growth from submission-equivalent steps to long run
# -----------------------------------------------------------------------

baseline = 'AbsMean+FP16 (original)'

print('(a) Mean error relative to FP16 baseline  (1.000x = same, lower = better)')
print(f'{"Steps":>7}  {"zero_frac":>9}', end='')
for name, _, _ in schemes:
    print(f'  {name:>25}', end='')
print()
print('-' * (20 + len(schemes) * 27))

for step in CHECKPOINTS:
    base_err = all_results[step][baseline][1]
    zf       = all_results[step][baseline][0]
    line     = f'{step:>7}  {zf:>9.4f}'
    for name, _, _ in schemes:
        me  = all_results[step][name][1]
        rel = me / base_err if base_err > 0 else 1.0
        line += f'  {rel:>12.4f}x ({me:.4f})'
    print(line)

print()
print('(b) Error growth: step 1000 (approx 10-min run equiv) -> step 50000')
step_lo, step_hi = 1000, 50000
for name, _, _ in schemes:
    e_lo  = all_results[step_lo][name][1]
    e_hi  = all_results[step_hi][name][1]
    zf_lo = all_results[step_lo][name][0]
    zf_hi = all_results[step_hi][name][0]
    g     = e_hi / e_lo if e_lo > 0 else float('inf')
    print(f'  {name:<28}  zf:{zf_lo:.3f}->{zf_hi:.3f}  err:{e_lo:.6f}->{e_hi:.6f}  ({g:.3f}x)')

(a) Mean error relative to FP16 baseline  (1.000x = same, lower = better)
  Steps  zero_frac    AbsMean+FP16 (original)     AbsMean+BF16 (current)       AbsMean+FP32 (ideal)             AbsMean-NoZero                   RMS+FP32
-----------------------------------------------------------------------------------------------------------------------------------------------------------
      0     0.2480        1.0000x (0.0095)        1.0000x (0.0095)        1.0000x (0.0095)        1.0803x (0.0102)        1.0003x (0.0095)
    100     0.2543        1.0000x (0.0103)        1.0000x (0.0103)        1.0000x (0.0103)        1.0591x (0.0109)        1.0026x (0.0103)
    500     0.2479        1.0000x (0.0116)        1.0000x (0.0116)        1.0000x (0.0116)        1.0556x (0.0123)        1.0039x (0.0117)
   1000     0.2329        1.0000x (0.0118)        1.0000x (0.0118)        1.0000x (0.0118)        1.0480x (0.0123)        1.0039x (0.0118)
   2500     0.2085        1.0000x (0.0119)        1.0000x (0

### 5. Theoretical Correction Factor Growth

In [5]:
# -----------------------------------------------------------------------
# For a ternary distribution with zero_frac z, the non-zero weights are
# split evenly between -1 and +1. The correction factors are:
#
#   AbsMean:  1 / q_absmean = 1 / (1 - z)        -- grows unboundedly
#   RMS:      1 / q_rms     = 1 / sqrt(1 - z)     -- grows slower
#   NoZero:   1.0                                  -- flat, always exact
#
# FP16 rounding error on the stored scale gets multiplied by these factors.
# -----------------------------------------------------------------------

print(f'{"zero_frac":>9} | {"AbsMean 1/(1-z)":>18} | {"RMS 1/sqrt(1-z)":>18} | {"NoZero 1.0":>12} | {"RMS/AbsMean":>12}')
print('-' * 82)

for z in [0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70]:
    am    = 1.0 / (1.0 - z)
    rms   = 1.0 / ((1.0 - z) ** 0.5)
    ratio = rms / am
    print(f'{z:>9.2f} | {am:>18.4f} | {rms:>18.4f} | {1.0:>12.4f} | {ratio:>12.4f}')

print()
print('At zero_frac=0.50: AbsMean amplifies stored-scale error 2x, RMS only 1.41x.')
print('At zero_frac=0.70: AbsMean amplifies 3.33x, RMS only 1.83x.')

zero_frac |    AbsMean 1/(1-z) |    RMS 1/sqrt(1-z) |   NoZero 1.0 |  RMS/AbsMean
----------------------------------------------------------------------------------
     0.10 |             1.1111 |             1.0541 |       1.0000 |       0.9487
     0.20 |             1.2500 |             1.1180 |       1.0000 |       0.8944
     0.25 |             1.3333 |             1.1547 |       1.0000 |       0.8660
     0.30 |             1.4286 |             1.1952 |       1.0000 |       0.8367
     0.35 |             1.5385 |             1.2403 |       1.0000 |       0.8062
     0.40 |             1.6667 |             1.2910 |       1.0000 |       0.7746
     0.45 |             1.8182 |             1.3484 |       1.0000 |       0.7416
     0.50 |             2.0000 |             1.4142 |       1.0000 |       0.7071
     0.60 |             2.5000 |             1.5811 |       1.0000 |       0.6325
     0.70 |             3.3333 |             1.8257 |       1.0000 |       0.5477

At zero_frac=0

### 6. Speed Cost of Each Scheme at Real Model Matrix Sizes

In [6]:
import torch
import time

torch.manual_seed(42)
device = torch.device('cuda')

# -----------------------------------------------------------------------
# Test all three forward-pass scale computations at the exact matrix sizes
# from the 10L 768d submission model.
# c_qkv fuses Q+K+V: 8h*96 + 2*(4kv*96) = 768+768=1536 out dim.
# -----------------------------------------------------------------------

GROUP_SIZE = 128
N_WARMUP   = 200
N_RUNS     = 2000

SHAPES = [
    ('c_qkv      [1536,768]',  (1536, 768)),
    ('attn_proj  [768, 768]',  (768,  768)),
    ('mlp_fc     [3072,768]',  (3072, 768)),
    ('mlp_proj   [768,3072]',  (768,  3072)),
]


def s_absmean(w_g):
    return w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)

def s_absmean_nozero(w_g):
    abs_w    = w_g.abs()
    nz_sum   = abs_w.sum(-1, keepdim=True)
    nz_count = (abs_w > 0).float().sum(-1, keepdim=True).clamp(min=1)
    return (nz_sum / nz_count).clamp(min=1e-8)

def s_rms(w_g):
    return w_g.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-8)


scale_fns = [
    ('AbsMean (current)',  s_absmean),
    ('AbsMean-NoZero',     s_absmean_nozero),
    ('RMS',                s_rms),
]

totals = {name: 0.0 for name, _ in scale_fns}

for shape_name, shape in SHAPES:
    W = torch.randn(*shape, device=device, dtype=torch.bfloat16)
    print(f'\n  {shape_name}')
    for fn_name, fn in scale_fns:
        for _ in range(N_WARMUP):
            _ = fn(W.reshape(-1, GROUP_SIZE))
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_RUNS):
            _ = fn(W.reshape(-1, GROUP_SIZE))
        torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) / N_RUNS * 1000
        totals[fn_name] += ms
        print(f'    {fn_name:<25} {ms:.4f} ms/call')

# 10L x 4 matrices per layer = 40 STE calls per step
# Actual submission step time = 91.8ms
STEP_MS   = 91.8
BUDGET_S  = 599
BASE_STEPS = 6530

print(f'\n  Per-step overhead vs AbsMean (actual step={STEP_MS}ms, budget={BUDGET_S}s):')
base_ms = totals['AbsMean (current)'] * 10
for fn_name, _ in scale_fns:
    per_step   = totals[fn_name] * 10
    overhead   = per_step - base_ms
    pct        = overhead / STEP_MS * 100
    steps_lost = int(overhead / STEP_MS * BASE_STEPS)
    print(f'    {fn_name:<25} ~{per_step:.3f} ms/step  {overhead:+.3f} ms  ({pct:+.2f}%)  ~{steps_lost:+d} steps')


  c_qkv      [1536,768]
    AbsMean (current)         0.0206 ms/call
    AbsMean-NoZero            0.0528 ms/call
    RMS                       0.0256 ms/call

  attn_proj  [768, 768]
    AbsMean (current)         0.0208 ms/call
    AbsMean-NoZero            0.0532 ms/call
    RMS                       0.0256 ms/call

  mlp_fc     [3072,768]
    AbsMean (current)         0.0210 ms/call
    AbsMean-NoZero            0.0532 ms/call
    RMS                       0.0256 ms/call

  mlp_proj   [768,3072]
    AbsMean (current)         0.0210 ms/call
    AbsMean-NoZero            0.0530 ms/call
    RMS                       0.0256 ms/call

  Per-step overhead vs AbsMean (actual step=91.8ms, budget=599s):
    AbsMean (current)         ~0.834 ms/step  +0.000 ms  (+0.00%)  ~+0 steps
    AbsMean-NoZero            ~2.121 ms/step  +1.287 ms  (+1.40%)  ~+91 steps
    RMS                       ~1.024 ms/step  +0.190 ms  (+0.21%)  ~+13 steps
